In [ ]:
# ==========================================
# 1. SETUP & DATA PREPARATION (จากโค้ดของคุณ)
# ==========================================
import re
import unicodedata
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import normalize
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from transformers import AutoTokenizer, AutoModel
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import coalesce  
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
df = pd.read_csv('AFNC_news_dataset_tf-2.csv')

# --- Thai Normalization ---
ZW = ''.join(['\u200B', '\u200C', '\u200D', '\uFEFF'])
def normalize_thai(s):
    if pd.isna(s): return None
    s = str(s).replace('\u00A0', ' ').translate({ord(ch): None for ch in ZW})
    s = unicodedata.normalize('NFC', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

df['ประเภทข่าว'] = df['ประเภทข่าว'].apply(normalize_thai)
df['หมวดหมู่ของข่าว'] = df['หมวดหมู่ของข่าว'].apply(normalize_thai)
df = df.dropna(subset=['หัวข้อข่าว', 'ประเภทข่าว']).reset_index(drop=True)

# Mapping
label2id = {c: i for i, c in enumerate(sorted(df['ประเภทข่าว'].unique()))}
id2label = {i: c for c, i in label2id.items()}
df['label_id'] = df['ประเภทข่าว'].map(label2id)

cat2id = {c: i for i, c in enumerate(sorted(df['หมวดหมู่ของข่าว'].unique()))}
id2cat = {i: c for c, i in cat2id.items()}
df['category_id'] = df['หมวดหมู่ของข่าว'].map(cat2id)

# ==========================================
# 2. BERT EMBEDDING (WangchanBERTa)
# ==========================================
model_WCB = "airesearch/wangchanberta-base-att-spm-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_WCB, use_fast=False)
lm_model = AutoModel.from_pretrained(model_WCB).to(device).eval()

@torch.no_grad()
def get_bert_embeddings(texts, batch_size=32):
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = [str(t) for t in texts[i:i+batch_size]]
        inputs = tokenizer(batch, truncation=True, padding=True, max_length=256, return_tensors='pt').to(device)
        outputs = lm_model(**inputs)
        mask = inputs['attention_mask'].unsqueeze(-1)
        emb = (outputs.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1)
        all_embs.append(emb.cpu().numpy())
    return np.vstack(all_embs)

# สร้าง Embedding เริ่มต้น
content_emb = get_bert_embeddings(df['หัวข้อข่าว'].tolist())
x_np = normalize(content_emb, axis=1, norm='l2')



KeyError: 'วันและเวลาที่เผยแพร่'

In [23]:
# ==========================================
# 3. BALANCING DATA (ฉบับแก้ไข KeyError)
# ==========================================
print("\n🔍 Checking Columns...")
print(f"ชื่อคอลัมน์ที่มีในเครื่องคุณ: {df.columns.tolist()}")

# พยายามหาคอลัมน์ที่มีคำว่า 'วัน' หรือ 'เวลา'
possible_date_cols = [c for c in df.columns if 'วัน' in c or 'เวลา' in c or 'date' in c.lower()]
if not possible_date_cols:
    print("⚠️ ไม่พบคอลัมน์วันที่! จะใช้วิธีสุ่มข้อมูลแทนการเรียงตามวันที่")
    date_col = None
else:
    date_col = possible_date_cols[0] # เลือกคอลัมน์แรกที่เจอ
    print(f"✅ ใช้คอลัมน์วันที่: '{date_col}'")

def convert_thai_date(date_str):
    if pd.isna(date_str): return pd.NaT
    s = str(date_str).strip().split(' ')[0] # เอาแค่ส่วนวันที่
    s = s.replace('/', '-') # ทำความสะอาดเบื้องต้น
    p = s.split('-')
    if len(p) == 3:
        # ถ้าปีเป็น พ.ศ. (เกิน 2400) ให้ลบ 543
        try:
            year = int(p[2])
            if year > 2400: p[2] = str(year - 543)
            # กรณีเจอวันที่แบบ DD-MM-YYYY
            return f"{p[2]}-{p[1]}-{p[0]}"
        except: return pd.NaT
    return pd.NaT

if date_col:
    df['parsed_date'] = pd.to_datetime(df[date_col].apply(convert_thai_date), errors='coerce')
    # ถ้าแปลงแล้วเป็น NaT หมดเลย ให้เลิกใช้คอลัมน์นี้
    if df['parsed_date'].isna().all():
        print("⚠️ ข้อมูลในคอลัมน์วันที่รูปแบบไม่ถูกต้อง ข้ามการเรียงลำดับ")
        sort_col = None
    else:
        sort_col = 'parsed_date'
else:
    sort_col = None

# --- ส่วนการทำ Balanced Dataset ---
idx_real = np.where(df['label_id'] == 0)[0]
idx_fake = np.where(df['label_id'] == 1)[0]
min_len = min(len(idx_real), len(idx_fake))

if sort_col:
    idx_real_bal = df.loc[idx_real].sort_values(sort_col, ascending=False).head(min_len).index
    idx_fake_bal = df.loc[idx_fake].sort_values(sort_col, ascending=False).head(min_len).index
else:
    idx_real_bal = idx_real[:min_len]
    idx_fake_bal = idx_fake[:min_len]

idx_balanced = np.concatenate([idx_real_bal, idx_fake_bal])
df_balanced = df.loc[idx_balanced].reset_index(drop=True)

# สำคัญ: ต้อง Slice x_np ให้ตรงกับ idx_balanced ด้วย
x_balanced = x_np[idx_balanced]
y_balanced = df_balanced['label_id'].values

print(f"✅ Balanced Dataset Ready: {len(df_balanced)} samples")


🔍 Checking Columns...
ชื่อคอลัมน์ที่มีในเครื่องคุณ: ['วันและเวลาที่เผยแพร่  ', 'ลิงค์ข่าว', 'หัวข้อข่าว', 'เนื้อหาข่าว', 'หน่วยงานที่ตรวจสอบ', 'ประเภทข่าว', 'หมวดหมู่ของข่าว', 'จำนวนเข้าชม', 'Hashtag', 'label_binary', 'label_id', 'category_id']
✅ ใช้คอลัมน์วันที่: 'วันและเวลาที่เผยแพร่  '
✅ Balanced Dataset Ready: 5744 samples


In [25]:
# ==========================================
# 4. GNN MODEL & GRAPH CONSTRUCTION
# ==========================================
class GCNNet(nn.Module):
    def __init__(self, num_node_features, num_classes, hidden_channels=256, dropout_rate=0.4):
        super().__init__()
        self.conv1 = GCNConv(num_node_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, num_classes)
        self.dropout_rate = dropout_rate

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        edge_weight = getattr(data, 'edge_attr', None)
        x = F.relu(self.conv1(x, edge_index, edge_weight=edge_weight))
        x = F.dropout(x, p=self.dropout_rate, training=self.training)
        x = self.conv2(x, edge_index, edge_weight=edge_weight)
        return x

# สร้าง kNN Graph
k = 10
nbrs = NearestNeighbors(n_neighbors=k, metric='cosine').fit(x_balanced)
dists, indices = nbrs.kneighbors(x_balanced)
src = np.repeat(np.arange(len(x_balanced)), k-1)
dst = indices[:, 1:].reshape(-1)
edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long)
edge_weight = torch.tensor(1.0 - dists[:, 1:].reshape(-1), dtype=torch.float)

data = Data(x=torch.tensor(x_balanced, dtype=torch.float), 
            y=torch.tensor(y_balanced, dtype=torch.long),
            edge_index=edge_index, edge_attr=edge_weight).to(device)

model_gnn = GCNNet(768, 2).to(device)



In [26]:
# ==========================================
# 5. [DEBUG] TRACE & ERROR ANALYSIS FUNCTION
# ==========================================
def trace_and_analyze_error(target_idx, data, model, df_source):
    model.eval()
    with torch.no_grad():
        logits = model(data)
        probs = torch.softmax(logits, dim=1)
        pred = logits.argmax(dim=1)[target_idx].item()
        actual = data.y[target_idx].item()
        
    print("\n" + "🔍" * 15)
    print(f"ERROR ANALYSIS & TRACE: NODE {target_idx}")
    print(f"Headline: {df_source.iloc[target_idx]['หัวข้อข่าว']}")
    print(f"Result: Actual={id2label[actual]} | Pred={id2label[pred]} | Conf={probs[target_idx][pred]:.4f}")
    print("🔍" * 15)

    # --- Step 1: BERT Word Importance ---
    inputs = tokenizer(df_source.iloc[target_idx]['หัวข้อข่าว'], return_tensors='pt', truncation=True).to(device)
    with torch.no_grad():
        out = lm_model(**inputs, output_attentions=True)
    # ดูความสำคัญของคำจาก Attention Head สุดท้าย
    attn = out.attentions[-1][0].mean(dim=0).mean(dim=0).cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    top_words = sorted(zip(tokens, attn), key=lambda x: x[1], reverse=True)[:5]
    print(f"\n[1] Word Importance (BERT):")
    for t, s in top_words: 
        if t not in ['<s>', '</s>', '<pad>']: print(f"   - {t:15} score: {s:.4f}")

    # --- Step 2: GNN Influence (Neighbors) ---
    row, col = data.edge_index
    neighbors = col[row == target_idx]
    neighbor_labels = [id2label[data.y[n].item()] for n in neighbors]
    neighbor_counts = Counter(neighbor_labels)
    
    print(f"\n[2] Neighbor Influence (Graph):")
    print(f"   - Total Neighbors: {len(neighbors)}")
    print(f"   - Label Distribution in Neighbors: {dict(neighbor_counts)}")
    
    if neighbor_counts[id2label[pred]] > neighbor_counts[id2label[actual]]:
        print("   🚩 Root Cause: 'Peer Pressure' - โดนกลุ่มเพื่อนดึงคะแนนไปทางคลาสที่ผิด")
    else:
        print("   🚩 Root Cause: 'Feature Ambiguity' - เนื้อหาข่าวเองมีความก้ำกึ่งสูง")

In [27]:
# ==========================================
# 6. RUN DEBUGGING
# ==========================================
# พยายามโหลด Weights (ถ้ามี)
try:
    model_gnn.load_state_dict(torch.load('best_model.pth', map_location=device))
    print("\n✅ Loaded best_model.pth")
except:
    print("\n⚠️ Weights not found, tracing with initial state.")

# เลือก Node แรกมาลอง Trace
trace_and_analyze_error(0, data, model_gnn, df_balanced)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.



✅ Loaded best_model.pth

🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍
ERROR ANALYSIS & TRACE: NODE 0
Headline: ไทยเรียกร้องให้กัมพูชารับผิดชอบ กรณียิงข้ามแดนเข้ามาฝั่งไทย จนทำให้กำลังพลได้รับบาดเจ็บ 1 นาย
Result: Actual=ข่าวจริง | Pred=ข่าวจริง | Conf=0.5299
🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍🔍


CamembertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.



[1] Word Importance (BERT):
   - รับผิดชอบ       score: 0.2254
   - เรียกร้อง       score: 0.2254
   - กัมพูชา         score: 0.0204

[2] Neighbor Influence (Graph):
   - Total Neighbors: 9
   - Label Distribution in Neighbors: {'ข่าวปลอม': 5, 'ข่าวจริง': 4}
   🚩 Root Cause: 'Feature Ambiguity' - เนื้อหาข่าวเองมีความก้ำกึ่งสูง


In [30]:
import torch
import numpy as np

def trace_with_formulas(node_idx, data, model, df_source):
    model.eval()
    headline = df_source.iloc[node_idx]['หัวข้อข่าว']
    
    print("\n" + "🔬" * 10 + " GNN DECISION LOGIC " + "🔬" * 10)
    print(f"HEADLINE: {headline}")
    print("=" * 60)

    with torch.no_grad():
        # --- ขั้นที่ 1: BERT (The Semantic Gauge) ---
        x_bert = data.x[node_idx]
        print(f"\n[STEP 1] BERT Embedding: 'การตีความเนื้อหา'")
        print(f"   📝 สูตร: Text -> Vector X (768 dimensions)")
        print(f"   💡 เกณฑ์: BERT จะมองหา 'คำสำคัญ' (Keywords) และ 'บริบท' (Context)")
        print(f"      ถ้าเจอคำที่มักอยู่ในข่าวปลอม เช่น 'ด่วนที่สุด', 'หลุดออกมา', 'สรรพคุณมหัศจรรย์'")
        print(f"      ตัวเลขใน Vector X จะขยับไปอยู่ในโซนของ 'ความไม่น่าเชื่อถือ'")

        # --- ขั้นที่ 2: Graph Influence (The Peer Review) ---
        row, col = data.edge_index
        neighbors = col[row == node_idx]
        neighbor_labels = [id2label[data.y[n].item()] for n in neighbors]
        
        print(f"\n[STEP 2] Graph Connectivity: 'กฎการรวมกลุ่ม'")
        print(f"   🤝 สูตร: kNN (Cosine Similarity) -> หาโหนดที่ใกล้กันที่สุด {len(neighbors)} อัน")
        print(f"   💡 เกณฑ์: 'คุณคือค่าเฉลี่ยของเพื่อนที่คุณคบ' (Homophily Property)")
        print(f"      โมเดลเชื่อว่าข่าวประเภทเดียวกันมักจะอ้างถึงแหล่งข่าวคล้ายกันหรือเขียนคล้ายกัน")
        print(f"      ในเคสนี้ เพื่อนบ้านของคุณคือ: {dict(Counter(neighbor_labels))}")

        # --- ขั้นที่ 3: GCN Layer 1 (Message Passing) ---
        w1 = model.conv1.lin.weight
        z1_me = torch.matmul(x_bert, w1.T) + model.conv1.bias
        
        neighbor_features = data.x[neighbors]
        z1_neighbors = torch.matmul(neighbor_features, w1.T) + model.conv1.bias
        z1_agg = torch.mean(z1_neighbors, dim=0) 
        
        print(f"\n[STEP 3] GCN Aggregation: 'การแลกเปลี่ยนข้อมูล'")
        print(f"   🌀 สูตร: h_v = ReLU( W * mean(h_u for u in neighbors) )")
        print(f"   💡 เกณฑ์: เป็นการ 'ถ่วงดุล' ระหว่าง สิ่งที่ตัวเองเป็น (BERT) กับ สิ่งที่สังคมรอบข้างบอก (Neighbors)")
        print(f"      ตัวเลข {z1_me[:3].cpu().numpy().round(3)} (จากตัวเอง) จะถูกนำไปผสมกับ")
        print(f"      ตัวเลข {z1_agg[:3].cpu().numpy().round(3)} (จากคนอื่น) เพื่อหาจุดกึ่งกลางของความจริง")

        # --- ขั้นที่ 4: Final Squeeze & Prediction ---
        logits = model(data)[node_idx]
        probs = torch.softmax(logits, dim=0)
        
        print(f"\n[STEP 4] Decision Output: 'การตัดสินสุดท้าย'")
        print(f"   ⚖️ สูตร: Softmax(z) = exp(z_i) / sum(exp(z_j))")
        print(f"   💡 เกณฑ์ตัดสิน: ใครได้แต้มมากกว่า (Score 0 vs Score 1) คือผู้ชนะ")
        print(f"      แต้มข่าวจริง: {probs[0]:.4f} | แต้มข่าวปลอม: {probs[1]:.4f}")

        print("\n" + "📊" * 5 + " สรุปเกณฑ์ที่โมเดลใช้ " + "📊" * 5)
        # วิเคราะห์ผลต่าง
        if neighbor_labels.count(id2label[data.y[node_idx].item()]) < len(neighbors)/2:
            print("🚩 ข่าวนี้ทายยาก: เพราะเนื้อหาข้างในดูเป็น 'ข่าวจริง' แต่คนรอบข้างส่วนใหญ่เป็น 'ข่าวปลอม'")
        else:
            print("✅ ข่าวนี้ทายง่าย: ทั้งเนื้อหาและกลุ่มเพื่อนไปในทิศทางเดียวกัน")

trace_with_formulas(0, data, model_gnn, df_balanced)


🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬 GNN DECISION LOGIC 🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬
HEADLINE: ไทยเรียกร้องให้กัมพูชารับผิดชอบ กรณียิงข้ามแดนเข้ามาฝั่งไทย จนทำให้กำลังพลได้รับบาดเจ็บ 1 นาย

[STEP 1] BERT Embedding: 'การตีความเนื้อหา'
   📝 สูตร: Text -> Vector X (768 dimensions)
   💡 เกณฑ์: BERT จะมองหา 'คำสำคัญ' (Keywords) และ 'บริบท' (Context)
      ถ้าเจอคำที่มักอยู่ในข่าวปลอม เช่น 'ด่วนที่สุด', 'หลุดออกมา', 'สรรพคุณมหัศจรรย์'
      ตัวเลขใน Vector X จะขยับไปอยู่ในโซนของ 'ความไม่น่าเชื่อถือ'

[STEP 2] Graph Connectivity: 'กฎการรวมกลุ่ม'
   🤝 สูตร: kNN (Cosine Similarity) -> หาโหนดที่ใกล้กันที่สุด 9 อัน
   💡 เกณฑ์: 'คุณคือค่าเฉลี่ยของเพื่อนที่คุณคบ' (Homophily Property)
      โมเดลเชื่อว่าข่าวประเภทเดียวกันมักจะอ้างถึงแหล่งข่าวคล้ายกันหรือเขียนคล้ายกัน
      ในเคสนี้ เพื่อนบ้านของคุณคือ: {'ข่าวปลอม': 5, 'ข่าวจริง': 4}

[STEP 3] GCN Aggregation: 'การแลกเปลี่ยนข้อมูล'
   🌀 สูตร: h_v = ReLU( W * mean(h_u for u in neighbors) )
   💡 เกณฑ์: เป็นการ 'ถ่วงดุล' ระหว่าง สิ่งที่ตัวเองเป็น (BERT) กับ สิ่งที่สังคมรอบข้างบอก (Neighbors)
   